# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [16]:
# imports
from IPython.display import Markdown, display, update_display
from openai import OpenAI

from dotenv import load_dotenv
import os


load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

API key looks good so far


In [17]:
# constants

MODEL_GPT = 'gpt-5-nano'
MODEL_OLLAMA = 'gemma4:e2b'


openai = OpenAI()
ollama = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")

In [18]:
# set up environment

def ask_question(question, client=openai):
    stream = client.chat.completions.create(
            model=MODEL_GPT if client == openai else MODEL_OLLAMA,
            messages=[
                {"role": "system", "content": "You are a helpful assistant that helps students with technical questions."},
                {"role": "user", "content": question}
            ],
            stream=True
        )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [19]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [15]:
# Get gpt-5-nano to answer, with streaming

ask_question(question)

Short answer:
- It builds a set of all non-empty author values from the iterable books, and then yields each of those authors one by one.
- In effect, you get a generator that yields each unique author (no duplicates) found in books, skipping missing or falsy authors.

What it does, step by step:
- {book.get("author") for book in books if book.get("author")}
  - Iterates over books (each item is expected to be a dict-like object).
  - For each book, it calls book.get("author") to get the author value (or None if the key is missing).
  - The if book.get("author") filter keeps only truthy author values (e.g., non-empty strings), discarding None, "", etc.
  - The set comprehension collects these author values and removes duplicates (sets have unique elements).
- yield from <that set>
  - Takes the resulting set and yields each element from it to the caller of the surrounding generator function.

Why you might write it this way:
- To produce a stream (generator) of unique authors, without duplicates, and ignoring missing authors.
- Using a set ensures deduplication in one pass.

Important nuances:
- The order of yielded authors is not guaranteed because sets are unordered.
- All authors are loaded into memory as a set before yielding; this has O(n) time and O(u) space complexity, where n is the number of books and u is the number of unique non-empty authors.
- If you need deterministic ordering, you’d need a different approach (e.g., sort the results or preserve insertion order with a seen set and streaming).

Example:
- Given books = [{"author": "Alice"}, {"author": "Bob"}, {"author": "Alice"}, {"author": ""}, {"title": "Untitled"}]
- The set comprehension picks {"Alice", "Bob"} (empty string and missing author are filtered out).
- yield from yields "Alice" and "Bob" in some order.

Common alternative if you need ordering:
- You could do: for author in sorted({book.get("author") for book in books if book.get("author")}): yield author
- Or stream with a seen set to preserve first-seen order.

In [21]:
# Get Llama 3.2 to answer

ask_question(question, ollama)

This single line of code is a powerful and efficient way to extract specific data from a collection and yield those data points, typically within a generator function.

Here is a detailed breakdown of what the code does and why it is structured this way.

---

## 1. What the Code Does (The Action)

The code is designed to **extract all unique author names from a list of books that actually have an author recorded, and yield them one by one.**

In simple terms: **It iterates through a collection of books and prints (yields) the names of the authors for every book it finds.**

## 2. How the Code Works (The Mechanics)

The line uses a combination of a **Set Comprehension** and the **`yield from`** statement.

Let's break down the expression component by component: `{book.get("author") for book in books if book.get("author")}`

### A. The Set Comprehension (The Inner Work)

```python
{book.get("author") for book in books if book.get("author")}
```

*   **`for book in books`**: This starts looping through every item (`book`) in the `books` collection (which is assumed to be a list of dictionaries).
*   **`if book.get("author")`**: This is a filter. It only proceeds if the value associated with the key `"author"` exists and is "truthy" (i.e., not `None`, not an empty string, etc.). This prevents errors if some books are missing an author.
*   **`book.get("author")`**: For every book that passes the filter, this expression extracts the actual author's name.
*   **`{ ... }`**: This collects all the extracted author names into a **Set**. Using a set automatically handles duplicate author names (if John Smith wrote two books, his name will only appear once in the resulting set).

### B. The Generator Statement (The Output)

```python
yield from (the set created above)
```

*   **`yield from`**: This is a feature used inside a function that defines a **generator**. It delegates the iteration directly to another iterable (in this case, the set we just created). Instead of manually writing a `for` loop and using `yield` inside it, `yield from` handles the entire iteration process, making the code much cleaner.

---

## 3. Why It Is Structured This Way (The Purpose)

This pattern is used for efficiency and specific programming paradigms, primarily related to **Generator Functions**.

### A. Efficiency (Lazy Evaluation)

The most important reason for using this pattern is **lazy evaluation**.

*   **Standard List Comprehension:** If you used a simple list comprehension (`[book.get("author") for book in books if book.get("author)]`), Python would immediately calculate and store *all* the author names in memory before returning the final list. This is fine for small lists, but inefficient for very large datasets.
*   **Generator/`yield from`:** By using a generator approach, the code does **not** calculate and store the entire result upfront. It calculates the next author's name *only when requested* (when the code iterates over the yielded results). This saves a massive amount of memory, especially when dealing with millions of books.

### B. Cleanliness and Readability

The `yield from` syntax is syntactic sugar that simplifies the process of iterating over an external sequence, making the intent of the code very clear: "Give me all the items from this sequence."

### Summary Table

| Code Part | Purpose | Benefit |
| :--- | :--- | :--- |
| **Set Comprehension** | Filters books and extracts unique author names. | Ensures accuracy and uniqueness. |
| **`yield from`** | Delegates iteration to the generated set. | Creates a memory-efficient **generator**. |
| **Overall Result** | Yields author names one at a time. | Saves memory by performing calculations on demand (lazy evaluation). |